# IN6227 Assignment 1 — Variant 1: Comparing Two Classifiers

**Course assignment:** IN6227-Assignment-1

This notebook is the reproducible companion to the two-column PDF report. It follows the familiar assignment-notebook sequence of **import → inspect data → prepare features → train → evaluate → discuss**, as seen in public [2025 IN6227 example A](https://github.com/FLAK813/IN6227-2025-Assignment1/blob/main/Assignment1.ipynb) and [example B](https://github.com/LITTLEWHALE000/IN6227-Data-Mining-Assignment/blob/main/IN6227_Assignment_1.ipynb). Their Adult/Census Income data and results are **not** used here: this assignment supplies different `train.csv` and `test.csv` files. All decisions and interpretations below are original to the supplied data.

**How to run:** install `requirements.txt`; place `train.csv` and `test.csv` in a local `dataset/` folder, or set the `IN6227_DATA_DIR` environment variable to their folder; run cells from top to bottom. The data are intentionally not published in the GitHub repository. The notebook's documented results were independently produced by `assignment1_analysis.py` with random seed 42; running all cells should reproduce them up to minor library-version differences.

## 1. Goal and evaluation plan

The target is binary (`yes` / `no`). I compare **logistic regression** with a **random forest** because the former gives a simple linear reference model and the latter can model nonlinear interactions. The comparison is more informative than evaluating only one model.

I use the supplied training file for a stratified 80/20 development split. Candidate settings are chosen **only on the validation part**. After selecting one setting per model family, I refit on all labeled training rows and evaluate the supplied test file once. This preserves the test file as an independent final check. Average precision (AP) is the primary selection metric because only about 24% of rows are `yes`; accuracy alone can hide poor detection of the less common class. I also report accuracy, balanced accuracy, precision, recall, F1, ROC-AUC and confusion matrices so the trade-offs are visible.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
DATA_DIR = Path(os.environ.get("IN6227_DATA_DIR", "dataset"))
if not (DATA_DIR / "train.csv").exists() and (DATA_DIR / "dataset" / "train.csv").exists():
    DATA_DIR = DATA_DIR / "dataset"
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
assert train.columns.tolist() == test.columns.tolist(), "Train and test schemas differ"
assert "label" in train.columns
feature_columns = [c for c in train.columns if c != "label"]
numeric = train[feature_columns].select_dtypes(include="number").columns.tolist()
categorical = [c for c in feature_columns if c not in numeric]
print(f"Raw train/test rows: {len(train):,} / {len(test):,}")
print(f"Predictors: {len(feature_columns)} ({len(numeric)} numeric, {len(categorical)} categorical)")
print("Numeric:", numeric)
print("Categorical:", categorical)

Raw train/test rows: 31,112 / 13,334
Predictors: 15 (7 numeric, 8 categorical)
Numeric: ['aptitude_score', 'performance_score', 'stability_index', 'load_ratio', 'index_weight', 'activity_duration', 'composite_rank']
Categorical: ['geological_era', 'weather_pattern', 'region', 'personal_interest', 'brew_preference', 'instrument', 'species', 'mineral_type']


## 2. Data inspection and cleaning decisions

The data have 31,112 training rows, 13,334 test rows and 15 predictors. Three training rows and one test row lack the **target label**. They cannot contribute to supervised fitting or scoring, so I exclude just those four rows. For missing **predictor** values, I keep the rows and impute within each training fold; dropping complete rows would discard information unnecessarily. I keep the provided train/test files separate throughout model selection.

The files contain no exact duplicate rows within either file. I therefore do not apply arbitrary deduplication. Numeric columns have long tails (for example `load_ratio`), but without feature definitions I cannot establish that extreme values are errors. I retain them rather than silently clipping values. The same caution applies to rare categorical levels. I do **not** create synthetic features: the meaning and collection timing of these variables are unspecified, and engineered combinations could be hard to justify or unavailable when making predictions.

In [2]:
train_clean = train.dropna(subset=["label"]).copy()
test_clean = test.dropna(subset=["label"]).copy()
assert set(train_clean["label"]) == {"no", "yes"}
assert set(test_clean["label"]) == {"no", "yes"}

summary = pd.DataFrame({
    "train": [len(train), len(train_clean), int(train[feature_columns].isna().sum().sum()),
              int(train.duplicated().sum()), float((train_clean["label"] == "yes").mean())],
    "test": [len(test), len(test_clean), int(test[feature_columns].isna().sum().sum()),
             int(test.duplicated().sum()), float((test_clean["label"] == "yes").mean())],
}, index=["raw rows", "labeled rows", "missing predictor cells", "exact duplicate rows", "yes prevalence"])
display(summary)
display(train[feature_columns].isna().sum().loc[lambda s: s > 0].to_frame("missing train cells"))
display(train[numeric].describe(percentiles=[0.25, 0.5, 0.75]).T[["min", "25%", "50%", "75%", "max"]])

                                train          test
raw rows                 31112.000000  13334.000000
labeled rows             31109.000000  13333.000000
missing predictor cells     47.000000     20.000000
exact duplicate rows         0.000000      0.000000
yes prevalence               0.239931      0.237606
                   missing train cells
geological_era                       2
weather_pattern                      3
region                               4
aptitude_score                       4
performance_score                    2
stability_index                      2
personal_interest                    6
load_ratio                           2
brew_preference                      7
instrument                           3
species                              3
index_weight                         3
mineral_type                         3
activity_duration                    2
composite_rank                       1
                      min       25%      50%        75%       ma

## 3. Features, split and leakage-safe preprocessing

I retain all 15 supplied predictors (7 numeric and 8 categorical). Logistic regression needs comparable numeric scales, so numeric missing values receive the **training-fold median** and numeric values are standardized. The median is less affected by long tails than the mean. Categorical missing values receive the **training-fold mode**, then categories are one-hot encoded. Unknown categories in validation or test are ignored rather than causing a failure. These transformations are inside a scikit-learn `Pipeline`; `fit` learns imputation values, scales and category lists only from the appropriate training data. Scaling is unnecessary for trees but harmless and allows a consistent preprocessing design.

The split is stratified because the positive class is less common. Seed 42 makes it repeatable. A single split is a reasonable small comparison under the short assignment format; repeated cross-validation would give more stable selection estimates, and is listed as a limitation below.

In [3]:
# pandas nullable string columns are converted to object so sklearn receives np.nan.
for frame in (train_clean, test_clean):
    frame[categorical] = frame[categorical].astype(object)
    frame[categorical] = frame[categorical].where(pd.notna(frame[categorical]), np.nan)

X = train_clean[feature_columns]
y = (train_clean["label"] == "yes").astype(int)
X_test = test_clean[feature_columns]
y_test = (test_clean["label"] == "yes").astype(int)
X_fit, X_val, y_fit, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

def preprocessing():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])

print(f"Development fit/validation rows: {len(X_fit):,} / {len(X_val):,}")
print(f"Positive prevalence: fit={y_fit.mean():.3%}, validation={y_val.mean():.3%}")

Development fit/validation rows: 24,887 / 6,222
Positive prevalence: fit=23.992%, validation=23.995%


## 4. Models and small validation search

I compare two model families and a small, declared set of settings rather than searching many combinations until one looks best by chance. Logistic regression uses L2 regularization and tests `C = 0.1, 1, 10` (larger `C` means weaker regularization). The random forest uses 150 trees, minimum leaf size 5, and either maximum depth 12 or no depth cap. A leaf minimum discourages very small, unstable terminal groups. Each candidate uses the **same split**. The candidate with highest validation AP is selected **within each family**; the test set is never used for that choice.

Scores at threshold 0.50 are supplementary. I do not tune that threshold on the test set. If the real application has a specified cost for missed `yes` cases, it should choose a threshold on validation data before test evaluation.

In [4]:
def summarize(y_true, probability):
    predicted = probability >= 0.5
    return {
        "accuracy": accuracy_score(y_true, predicted),
        "balanced_accuracy": balanced_accuracy_score(y_true, predicted),
        "precision_yes": precision_score(y_true, predicted, zero_division=0),
        "recall_yes": recall_score(y_true, predicted, zero_division=0),
        "f1_yes": f1_score(y_true, predicted, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "confusion_matrix_no_yes": confusion_matrix(y_true, predicted).tolist(),
    }

candidates = {
    "logistic_C_0.1": LogisticRegression(C=0.1, max_iter=1000, solver="lbfgs", random_state=SEED),
    "logistic_C_1": LogisticRegression(C=1, max_iter=1000, solver="lbfgs", random_state=SEED),
    "logistic_C_10": LogisticRegression(C=10, max_iter=1000, solver="lbfgs", random_state=SEED),
    "forest_depth_12": RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=SEED
    ),
    "forest_depth_none": RandomForestClassifier(
        n_estimators=150, max_depth=None, min_samples_leaf=5, n_jobs=-1, random_state=SEED
    ),
}
validation = {}
for name, classifier in candidates.items():
    pipe = Pipeline([("prep", preprocessing()), ("classifier", classifier)])
    pipe.fit(X_fit, y_fit)
    validation[name] = summarize(y_val, pipe.predict_proba(X_val)[:, 1])

validation_table = pd.DataFrame(validation).T.drop(columns="confusion_matrix_no_yes")
display(validation_table.round(4))
selections = {
    family: max((name for name in candidates if name.startswith(prefix)),
                key=lambda name: validation[name]["average_precision"])
    for family, prefix in [("logistic_regression", "logistic"),
                           ("random_forest", "forest")]
}
print("Selected by validation AP:", selections)

                   accuracy balanced_accuracy precision_yes recall_yes    f1_yes   roc_auc average_precision
logistic_C_0.1     0.833173          0.736711      0.691016   0.551239  0.613264  0.885553          0.696288
logistic_C_1       0.834137          0.740324      0.690339   0.559946  0.618343  0.886337          0.697381
logistic_C_10       0.83269           0.74006      0.684339   0.561956  0.617139   0.88652          0.697749
forest_depth_12    0.828512          0.714853      0.701705   0.496316  0.581404   0.88396          0.693075
forest_depth_none  0.828672          0.726188      0.685169   0.529136  0.597128  0.884436          0.696627
Selected by validation AP: {'logistic_regression': 'logistic_C_10', 'random_forest': 'forest_depth_none'}


**Recorded validation finding (seed 42):** Logistic AP rises from about 0.696 to 0.698 as `C` increases from 0.1 to 10, so `C=10` is selected. Forest AP is about 0.693 at depth 12 and 0.697 with no depth cap, so the latter is selected. The differences are small, which argues against strong claims about hyperparameter superiority.

### Feature-availability sensitivity: `composite_rank`

The name `composite_rank` suggests a derived feature, but the data provide no provenance or collection time. If it is unavailable when predictions are made, using it would be inappropriate. I therefore repeat the **already selected** model configurations on the same validation split after removing this column. This is a sensitivity check, not another opportunity to tune the test result. The ablation does not establish whether the remaining predictors would be available in deployment.

In [5]:
without_rank = [c for c in feature_columns if c != "composite_rank"]
numeric_without = [c for c in numeric if c != "composite_rank"]
categorical_without = [c for c in categorical if c != "composite_rank"]
preprocessor_without_rank = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_without),
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_without),
])
sensitivity_rows = []
for family, selected_name in selections.items():
    pipe = Pipeline([("prep", preprocessor_without_rank),
                     ("classifier", candidates[selected_name])])
    pipe.fit(X_fit[without_rank], y_fit)
    probability = pipe.predict_proba(X_val[without_rank])[:, 1]
    sensitivity_rows.append({
        "model": family,
        "AP all 15": validation[selected_name]["average_precision"],
        "AP without composite_rank": average_precision_score(y_val, probability),
    })
sensitivity_table = pd.DataFrame(sensitivity_rows).set_index("model")
sensitivity_table["AP change"] = (sensitivity_table["AP without composite_rank"]
                                  - sensitivity_table["AP all 15"])
display(sensitivity_table.round(5))

                     AP all 15  AP without composite_rank  AP change
model                                                               
logistic_regression    0.69775                    0.69769   -0.00006
random_forest          0.69663                    0.69618   -0.00044


**Interpretation:** In the verified run, removing `composite_rank` changed validation AP by less than 0.0005 for either model. That small change suggests the observed comparison is not driven by this one feature on this split. It does **not** prove the feature is available at prediction time, nor that the result is stable in new data.

## 5. Refit and evaluate the untouched supplied test file

Once settings are frozen, I refit each pipeline on all labeled training rows. The supplied test file is scored at a fixed 0.50 threshold. The reported AP and ROC-AUC use probabilities and therefore do not depend on this threshold. The `no`-only baseline achieves about 76.2% test accuracy while detecting zero `yes` cases; that is why accuracy by itself is insufficient.

In [6]:
held_out = {}
test_probabilities = {}
for family, selected_name in selections.items():
    pipe = Pipeline([("prep", preprocessing()),
                     ("classifier", candidates[selected_name])])
    pipe.fit(X, y)
    probability = pipe.predict_proba(X_test)[:, 1]
    test_probabilities[family] = probability
    held_out[family] = summarize(y_test, probability)

test_table = pd.DataFrame(held_out).T.drop(columns="confusion_matrix_no_yes")
display(test_table.round(4))
print(f"Always-no accuracy: {(y_test == 0).mean():.4f}; yes recall: 0")
for family, result in held_out.items():
    print(f"{family} confusion matrix (true rows / predicted columns: no, yes):")
    display(pd.DataFrame(result["confusion_matrix_no_yes"],
                         index=["true no", "true yes"],
                         columns=["pred no", "pred yes"]))

                     accuracy balanced_accuracy precision_yes recall_yes    f1_yes   roc_auc average_precision
logistic_regression  0.839571          0.747253      0.698572   0.571338  0.628581  0.888825          0.702472
random_forest        0.839271          0.745101      0.700274   0.565657  0.625808  0.889711          0.708431
Always-no accuracy: 0.7624; yes recall: 0
logistic_regression confusion matrix (true rows / predicted columns: no, yes):
          pred no  pred yes
true no      9384       781
true yes     1358      1810
random_forest confusion matrix (true rows / predicted columns: no, yes):
          pred no  pred yes
true no      9398       767
true yes     1376      1792


**Recorded test finding:** Logistic regression: accuracy 0.8396, positive recall 0.5713, F1 0.6286, AP 0.7025. Random forest: accuracy 0.8393, positive recall 0.5657, F1 0.6258, AP 0.7084. Both have nearly identical accuracy. Logistic regression finds 18 more positive cases at threshold 0.50 (1,810 versus 1,792), while the forest has slightly higher AP, which evaluates ranking across thresholds. Both miss more than 1,350 of the 3,168 actual positive cases; the operational cost of those misses matters more than a tiny accuracy difference.

## 6. Uncertainty in the AP difference

The forest-minus-logistic AP gap is about 0.006. I use 1,000 **paired bootstrap** resamples of test rows to describe its sampling uncertainty: each resample uses the same selected rows for both models, preserving the paired comparison. This interval is descriptive, not a new model-selection step, and it does not capture uncertainty from selecting hyperparameters on one validation split. The original test observations remain unchanged.

In [7]:
rng = np.random.default_rng(SEED)
y_array = y_test.to_numpy()
paired_differences = []
for _ in range(1000):
    sample = rng.integers(0, len(y_array), len(y_array))
    if len(np.unique(y_array[sample])) < 2:
        continue
    paired_differences.append(
        average_precision_score(y_array[sample], test_probabilities["random_forest"][sample])
        - average_precision_score(y_array[sample], test_probabilities["logistic_regression"][sample])
    )
interval = np.quantile(paired_differences, [0.025, 0.975])
gap = held_out["random_forest"]["average_precision"] - held_out["logistic_regression"]["average_precision"]
print(f"Forest - logistic test AP: {gap:.4f}")
print(f"Paired bootstrap 95% interval: [{interval[0]:.4f}, {interval[1]:.4f}]")

Forest - logistic test AP: 0.0060
Paired bootstrap 95% interval: [-0.0015, 0.0132]


**Interpretation:** The recorded interval is approximately **[-0.0015, 0.0132]**. It includes zero, so these data do not identify a clear AP winner. This does not prove the models are equivalent; it means the observed advantage is too small to claim confidently from this comparison.

## 7. Discussion, limitations and decision

1. **Which model would I use?** With no stated error costs, neither is a decisive winner. Logistic regression is a sensible starting choice because it is simpler and has slightly better positive recall/F1 at threshold 0.50. The forest has a small AP advantage that might matter if ranking candidates is the application. A final choice requires the application's false-positive/false-negative costs and verified feature availability.
2. **No test-set tuning.** Test labels are used only for final comparison and the descriptive uncertainty interval. Changing candidates or threshold after inspecting test scores would invalidate its role as an untouched estimate.
3. **One validation split.** Repeated stratified cross-validation would assess whether the small validation differences persist across splits. It was not run, so this notebook does not claim that stability.
4. **Unknown provenance.** Feature names do not establish how or when predictors were collected. For example, a derived field such as `composite_rank` may be unavailable at prediction time; deployment would need a provenance check.
5. **No external benchmark claim.** Public 2023/2025 IN6227 notebooks use the Adult dataset, different cleaning and splits. Their accuracy numbers cannot be compared numerically with this dataset's accuracy.

**Conclusion:** Both models reach about 84% accuracy on the supplied test file, but the minority-class recall is only about 57%. The small AP difference does not support declaring a winner. The reproducible validation-to-test workflow and clear treatment of the less common `yes` class are the main findings.